In [1]:
#Universal Sentence Encoder (USE) - DAN

import pandas as pd
import re
from nltk.corpus import stopwords
import tensorflow_hub as hub
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import nltk

# Stopwords setini indir
nltk.download('stopwords')
STOPWORDS = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\yusuf\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
# Remove special characters, punctuation, and convert to lowercase
def preprocess_text(text):
    if not isinstance(text, str):
        return ''  # Return empty string for NaN or non-string values
    text = re.sub(r'[^\w\s]', '', text.lower())  # Remove punctuation and convert to lowercase
    text = ' '.join(word for word in text.split() if word not in STOPWORDS)  # Remove stopwords
    return text

In [3]:
# Load the dataset
file_path = "filtered_file_2.xlsx"  # Replace with your actual file path
df = pd.read_excel(file_path)

# Apply text preprocessing
df['Abstract_clean'] = df['Abstract'].apply(preprocess_text)
df['Research_Areas_clean'] = df['Research Areas'].apply(preprocess_text)

# Combine text columns for analysis
df['combined_text'] = df['Abstract_clean'] + ' ' + df['Research_Areas_clean']


In [7]:
# User query
user_input = "nlp machine learning letters words models english turkish automata"
user_input_clean = preprocess_text(user_input)

# Load Universal Sentence Encoder (DAN)
use_model = hub.load("https://tfhub.dev/google/universal-sentence-encoder/4")

# Encode dataset and user query
print("Encoding dataset...")
corpus_embeddings = use_model(df['combined_text'].tolist())  # Encode all texts
query_embedding = use_model([user_input_clean])  # Encode user query

# Compute cosine similarity
print("Computing similarity scores...")
similarities = cosine_similarity(query_embedding, corpus_embeddings).flatten()

# Add similarity scores to the dataframe
df['similarity'] = similarities

# Calculate article count per researcher
df['article_count'] = df['Researcher Ids'].str.split(';').apply(len)

Encoding dataset...
Computing similarity scores...


In [8]:
# Flatten researchers for grouping
def split_researchers(row):
    if not isinstance(row['Researcher Ids'], str):
        return pd.DataFrame()
    researchers = row['Researcher Ids'].split(';')
    data = []
    for researcher in researchers:
        data.append({
            'Researcher': researcher.strip(),
            'combined_text': row['combined_text'],
            'Publication Year': row['Publication Year'],
            'similarity': row['similarity'],
            'article_count': row['article_count']
        })
    return pd.DataFrame(data)

flattened_df = pd.concat(df.apply(split_researchers, axis=1).to_list(), ignore_index=True)

# Group by researcher and calculate average similarity
grouped_df = flattened_df.groupby('Researcher', as_index=False).agg({
    'similarity': 'mean',
    'article_count': 'sum',
    'Publication Year': 'max'
})

# Count the number of articles per researcher
article_counts = flattened_df['Researcher'].value_counts()
grouped_df['article_count'] = grouped_df['Researcher'].map(article_counts)

# Normalize similarity scores based on the number of articles
k = 0.184  # Growth rate, calculated to make f(25) approximately 1
grouped_df['normalized_similarity'] = grouped_df['similarity'] * (1 - np.exp(-k * grouped_df['article_count']))

# Scale year weights for stronger impact (higher weight for recent years)
# Year set to 2026 because the articles published 2025 are being 2026 in the dataset
current_year = 2026
year_diff = current_year - grouped_df['Publication Year']
a=1
grouped_df['year_weight'] =  0.75 + 0.2 * (1 - 1 / (1 + np.exp(-a * (year_diff - 5))))

# Combine normalized similarity and year weight into a final score
grouped_df['final_score'] = grouped_df['normalized_similarity'] * grouped_df['year_weight']

# Get top researchers based on final score
top_researchers = grouped_df.sort_values(by='final_score', ascending=False).head(30)
print(top_researchers[['Researcher', 'final_score', 'article_count', 'year_weight', 'Publication Year']])

                                   Researcher  final_score  article_count  \
1027               Alpkocak, Adil/F-3388-2013     0.283516             19   
2626                Birant, Derya/U-6211-2017     0.257717             66   
16262            Varlıklar, Özlem/P-2441-2019     0.251991              9   
14182               Selver, Alper/V-5118-2019     0.250115             42   
7951               Işık, Zerrin/ABG-9392-2021     0.246606             22   
11488           Nasibov, Efendi/ISV-0173-2023     0.239538             46   
6645                Golcuk, Ilker/B-2116-2015     0.238781             19   
18816             kuntalp, mehmet/W-4067-2017     0.236975             14   
15871         Tüysüzoğlu, Göksu/HJH-7694-2023     0.228070              9   
2627        Birant, Kökten Ulaş/AAD-8386-2022     0.228049             16   
12293           Ozsoydan, Fehmi/AAC-5530-2019     0.222970             32   
15913               UTKU, Semih/LGY-2879-2024     0.222500             21   